In [14]:
import subprocess

# Get output of !pip install -q transformers datasets peft accelerate evaluate
result = subprocess.run(['pip', 'install', '-q', 'transformers', 'datasets', 'peft', 'accelerate', 'evaluate'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

# Upgrade peft and torchao to ensure compatible versions
result = subprocess.run(['pip', 'install', '--upgrade', 'peft'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)

result = subprocess.run(['pip', 'install', '--upgrade', 'torchao'], capture_output=True, text=True)
print(result.stdout)
print(result.stderr)





   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 37.1 MB/s eta 0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0




In [2]:
import pandas as pd
import torch

from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForMultipleChoice,
    Trainer,
    TrainingArguments
)

from peft import (
    LoraConfig,
    TaskType,
    get_peft_model
)

from torch.nn.functional import softmax

In [3]:
train = pd.read_csv("train.csv")

label_map = {
    "A":0,
    "B":1,
    "C":2,
    "D":3,
    "E":4
}

train["label"] = train["answer"].map(label_map)

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased"
)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [4]:
q1 = train.loc[150,"label"]

print("Q1 =",q1)

Q1 = 2


In [5]:
row=train.iloc[0]

formatted = str(row["prompt"])+" [SEP] "+str(row["B"])

q2=len(formatted)

print("Q2 =",q2)

Q2 = 407


In [6]:
choices=[]

for opt in ["A","B","C","D","E"]:

    choices.append(
        str(row["prompt"])
        +" [SEP] "
        +str(row[opt])
    )

enc = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

input_ids = enc["input_ids"].reshape(1,5,128)

print(input_ids.shape)

q3=input_ids.shape[1]

print("Q3 =",q3)

torch.Size([1, 5, 128])
Q3 = 5


In [7]:
all_ids=[]

for i in range(16):

    row=train.iloc[i]

    choices=[]

    for opt in ["A","B","C","D","E"]:

        choices.append(
            str(row["prompt"])
            +" [SEP] "
            +str(row[opt])
        )

    ids=tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
        return_tensors="pt"
    )["input_ids"]

    all_ids.append(ids)

batch=torch.stack(all_ids)

print(batch.shape)

q4=batch.numel()

print("Q4 =",q4)

torch.Size([16, 5, 128])
Q4 = 10240


In [8]:
model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

outputs=model(
    input_ids=input_ids,
    attention_mask=enc["attention_mask"].reshape(1,5,128)
)

print(outputs.logits.shape)

q5=outputs.logits.shape[1]

print("Q5 =",q5)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 5])
Q5 = 5


In [9]:
label=torch.tensor([train.loc[0,"label"]])

outputs=model(
    input_ids=input_ids,
    attention_mask=enc["attention_mask"].reshape(1,5,128),
    labels=label
)

print(outputs.loss.shape)

q6=outputs.loss.ndim

print("Q6 =",q6)

torch.Size([])
Q6 = 0


In [16]:
config=LoraConfig(

    r=8,

    lora_alpha=16,

    target_modules=[
        "query",
        "value"
    ],

    lora_dropout=0.1,

    bias="none",

    task_type=TaskType.SEQ_CLS
)

lora_model=get_peft_model(
    model,
    config
)

trainable=sum(
    p.numel()
    for p in lora_model.parameters()
    if p.requires_grad
)

print("Q7 =",trainable)

Q7 = 295681


In [13]:
def preprocess(example):

    choices=[]

    for opt in ["A","B","C","D","E"]:

        choices.append(
            str(example["prompt"])
            +" [SEP] "
            +str(example[opt])
        )

    enc=tokenizer(

        choices,

        padding="max_length",

        truncation=True,

        max_length=128
    )

    enc["labels"]=label_map[
        example["answer"]
    ]

    return enc


dataset=Dataset.from_pandas(
    train.iloc[:100]
)

dataset=dataset.map(preprocess)

print(
    len(dataset[0]["input_ids"])
)

q8=len(
    dataset[0]["input_ids"]
)

print("Q8 =",q8)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

5
Q8 = 5


In [17]:
small=Dataset.from_pandas(
    train.iloc[:32]
)

small=small.map(preprocess)

args=TrainingArguments(

    output_dir="tmp",

    max_steps=4,

    per_device_train_batch_size=4,

    gradient_accumulation_steps=1,

    logging_steps=1,

    report_to="none"
)

trainer=Trainer(

    model=lora_model,

    args=args,

    train_dataset=small
)

trainer.train()

print(
    "Q9 =",
    trainer.state.global_step
)

Map:   0%|          | 0/32 [00:00<?, ? examples/s]

[transformers] `use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,1.729802
2,1.672924
3,1.620798
4,1.627341


Q9 = 4


In [19]:
row=train.iloc[0]

choices=[]

for opt in ["A","B","C","D","E"]:

    choices.append(
        str(row["prompt"])
        +" [SEP] "
        +str(row[opt])
    )

enc=tokenizer(

    choices,

    padding="max_length",

    truncation=True,

    max_length=64,

    return_tensors="pt"
)

with torch.no_grad():

    logits=lora_model(

        input_ids=enc["input_ids"].unsqueeze(0).to(lora_model.device),

        attention_mask=enc["attention_mask"].unsqueeze(0).to(lora_model.device)

    ).logits


prob=softmax(
    logits,
    dim=1
)

q10=prob[0,4].item()

print(
    "Q10 =",
    round(q10,4)
)

Q10 = 0.1997
